## SQL queries 

In [8]:
import pandas as pd
import sqlite3
conn = sqlite3.connect("../data/customer_churn.db")
cursor = conn.cursor()

### Top 10 customers by total spending

In [3]:
query = """
SELECT
    customer_id,
    monthly_spend
FROM customers
ORDER BY monthly_spend DESC
LIMIT 10;
"""
cursor.execute(query)

results = cursor.fetchall()

top_customers = pd.DataFrame(
    results,
    columns=["customer_id", "monthly_spend"]
)

print("Top 10 Customers by Monthly Spending")
display(top_customers)

Top 10 Customers by Monthly Spending


,customer_id,monthly_spend
0,130477,40000.0
1,131886,40000.0
2,179556,40000.0
3,149731,40000.0
4,106118,40000.0
5,110050,40000.0
6,187099,40000.0
7,153012,40000.0
8,155381,40000.0
9,129248,40000.0


### 2. Monthly revenue. 

In [4]:
query = """
SELECT
    strftime('%Y-%m', last_order_date) AS month,
    ROUND(SUM(monthly_spend), 2) AS monthly_revenue
FROM customers
GROUP BY strftime('%Y-%m', last_order_date)
ORDER BY month;
"""

cursor.execute(query)

results = cursor.fetchall()

monthly_revenue = pd.DataFrame(
    results,
    columns=["month", "monthly_revenue"]
)

print("Monthly Revenue")
display(monthly_revenue)

Monthly Revenue


,month,monthly_revenue
0,2021-01,10980.02
1,2021-02,6293.26
2,2023-08,1085.65
3,2023-09,4168.28
4,2024-01,554.77
5,2024-02,3272.93
6,2024-08,893.00
7,2024-10,815.32
8,2024-11,6757.14
9,2024-12,1871923.09


### 3. Customers with No Orders in the Last 90 Days

In [5]:
query = """
SELECT
    customer_id,
    last_order_date,
    recency_days
FROM customers
WHERE recency_days > 90
ORDER BY recency_days DESC;
"""

cursor.execute(query)

results = cursor.fetchall()

inactive_customers = pd.DataFrame(
    results,
    columns=[
        "customer_id",
        "last_order_date",
        "recency_days"
    ]
)

print("Customers with No Orders in the Last 90 Days")
print("Count:", len(inactive_customers))
display(inactive_customers.head(10))

Customers with No Orders in the Last 90 Days
Count: 53235


,customer_id,last_order_date,recency_days
0,198019,2021-01-17 00:00:00,1808
1,167597,2021-02-23 00:00:00,1771
2,154669,2023-08-23 00:00:00,860
3,193421,2023-09-30 00:00:00,822
4,130080,2024-01-08 00:00:00,722
5,126733,2024-02-07 00:00:00,692
6,152958,2024-08-04 00:00:00,513
7,127750,2024-10-10 00:00:00,446
8,101396,2024-11-18 00:00:00,407
9,188826,2024-12-31 00:00:00,364


### 4. Churn Rate by City

In [6]:
query = """
SELECT
    city,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned_customers,
    ROUND(
        100.0 * SUM(churn) / COUNT(*),
        2 
    ) AS churn_rate
FROM customers
GROUP BY city
ORDER BY churn_rate DESC;
"""

cursor.execute(query)

results = cursor.fetchall()

churn_by_city = pd.DataFrame(
    results,
    columns=[
        "city",
        "total_customers",
        "churned_customers",
        "churn_rate"
    ]
)

print("\nChurn Rate by City")
display(churn_by_city)


Churn Rate by City


,city,total_customers,churned_customers,churn_rate
0,Kolkata,9941,2997,30.15
1,Ahmedabad,9859,2867,29.08
2,Bangalore,13845,4001,28.90
3,Chennai,10901,2979,27.33
4,Pune,10994,2993,27.22
5,Unknown,1000,263,26.30
6,Delhi,14874,3809,25.61
7,Mumbai,16783,4269,25.44
8,Hyderabad,11803,2982,25.26


### 5. Average Spending: Churned vs Non-Churned

In [7]:
query = """
SELECT
    churn,
    COUNT(*) AS customer_count,
    ROUND(AVG(monthly_spend), 2) AS average_monthly_spend
FROM customers
GROUP BY churn
ORDER BY churn;
"""

cursor.execute(query)

results = cursor.fetchall()

spending_by_churn = pd.DataFrame(
    results,
    columns=[
        "churn",
        "customer_count",
        "average_monthly_spend"
    ]
)

print("\nAverage Spending — Churned vs Non-Churned")
display(spending_by_churn)


Average Spending — Churned vs Non-Churned


,churn,customer_count,average_monthly_spend
0,0,72840,4820.99
1,1,27160,2623.79


### 6. Top Segments Associated with Churn

In [8]:
query = """
SELECT
    subscription_plan,
    acquisition_channel,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned_customers,
    ROUND(
        100.0 * SUM(churn) / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY
    subscription_plan,
    acquisition_channel
HAVING COUNT(*) >= 100
ORDER BY churn_rate DESC
LIMIT 10;
"""

cursor.execute(query)

results = cursor.fetchall()

top_segments = pd.DataFrame(
    results,
    columns=[
        "subscription_plan",
        "acquisition_channel",
        "total_customers",
        "churned_customers",
        "churn_rate"
    ]
)

print("\nTop Customer Segments Associated with Churn")
display(top_segments)


Top Customer Segments Associated with Churn


,subscription_plan,acquisition_channel,total_customers,churned_customers,churn_rate
0,Basic,Paid Ads,12052,4398,36.49
1,Basic,Social Media,8037,2728,33.94
2,Basic,Email Campaign,5069,1613,31.82
3,Basic,Organic,16178,5065,31.31
4,Basic,Referral,8996,2597,28.87
5,Standard,Paid Ads,8335,2361,28.33
6,Standard,Social Media,5494,1396,25.41
7,Standard,Email Campaign,3429,848,24.73
8,Standard,Organic,11199,2691,24.03
9,Standard,Referral,6313,1346,21.32
